# RAG as subagent

# Setup

In [ ]:
from typing import Any

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.tools.base import Tool
from conversational_toolkit.chunking.base import Chunk
from conversational_toolkit.llms.openai import OpenAILLM

from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever
from conversational_toolkit.agents.tool_agent import ToolAgent, QueryWithContext

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_llm,
    build_vector_store,
    VS_PATH,
    EMBEDDING_MODEL,
)

In [ ]:
chunks = load_chunks(max_files=5)
embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
db_chroma = await build_vector_store(
    chunks, embedding_model, db_path=VS_PATH, reset=False
)
vector_store = VectorStoreRetriever(embedding_model, db_chroma, top_k=5)

In [ ]:
llm = build_llm(backend="openai")

In [ ]:
def chunks_to_text(chunks: list[Chunk]) -> str:
    text = ""

    for chunk in chunks:
        text += (
            f"## Chunk {chunk.title}:\n```\n{chunk.content}\n```\n" + "-" * 30 + "\n\n"
        )

    text = text[:-4]

    return text

In [ ]:
class RetrieveRelevantChunks(Tool):
    def __init__(
        self, name: str, description: str, parameters: dict[str, Any], retriever
    ):
        self.name = name
        self.description = description
        self.parameters = parameters
        self.retriever = retriever

    async def call(self, args: dict[str, Any]) -> dict[str, Any]:
        query_with_history = args.get("query")

        retrieved = [await self.retriever.retrieve(q) for q in [query_with_history]]

        retrieved_as_text = [chunks_to_text(r) for r in retrieved]

        return {"result": retrieved_as_text}

In [ ]:
retriever_tool = RetrieveRelevantChunks(
    name="retrieve_relevant_chunks",
    description="Retrieves the most relevant chunks based on a query.",
    # What parameters it expects
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The query to retrieve relevant chunks for.",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    retriever=vector_store,
)

# Setup RAG subagent

In [ ]:
llm = OpenAILLM(tool_choice="auto", tools=[retriever_tool])

# Define the prompt
prompt = "You are a helpful assistant, answer shortly. Use the tools only when they are relevant, but if you do so, trust the results from the tools and use them in your answer, cite them precisely if you use them."
prompt_as_message = LLMMessage(
    content=[MessageContent(type="text", text=prompt)], role=Roles.SYSTEM
)

In [ ]:
rag_agent = ToolAgent(system_prompt=prompt, llm=llm, max_steps=5)

# Define it as a tool

In [ ]:
class RAGAgentAsTool(Tool):
    def __init__(
        self,
        name: str,
        description: str,
        parameters: dict[str, Any],
    ):
        self.name = name
        self.description = description
        self.parameters = parameters

    async def call(self, args: dict[str, Any]) -> dict[str, Any]:
        query = args.get("query")

        answer = await rag_agent.answer(
            query_with_context=QueryWithContext(query=query, history=[])
        )
        answer_as_text = str(answer.content)

        return {"result": answer_as_text}


rag_agent_as_tool_tool = RAGAgentAsTool(
    name="rag_agent_as_tool",
    description="Uses the RAG agent to answer queries about pellets, boxes and so on.",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The query to retrieve relevant chunks for.",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
)

# Define the Main Agent


In [ ]:
llm_main_agent = OpenAILLM(tool_choice="auto", tools=[rag_agent_as_tool_tool])

# Define the prompt, no need to explain the sources
prompt_main_agent = "You are a helpful assistant, answer shortly."
prompt_as_message_main_agent = LLMMessage(
    content=[MessageContent(type="text", text=prompt_main_agent)], role=Roles.SYSTEM
)

main_agent = ToolAgent(system_prompt=prompt_main_agent, llm=llm_main_agent, max_steps=5)

# Test the agent

## First Simple Question

In [ ]:
conversation = [prompt_as_message_main_agent]

In [ ]:
query = "What is a Einstein theory of relativity in the context of physics?"
query_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query)], role=Roles.USER
)
query_with_context = QueryWithContext(query=query, history=[])

answer = await main_agent.answer(query_with_context)

In [ ]:
print(answer.content[0].text)

In [ ]:
conversation += [query_as_message, answer]

## Our Question

In [ ]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query)], role=Roles.USER
)
query_with_context = QueryWithContext(query=query, history=conversation)

answer = await main_agent.answer(query_with_context)

In [ ]:
print(answer.content[0].text)

In [ ]:
conversation += [query_as_message, answer]

# Test Memory

In [ ]:
query = "Summarize the conversation (some words per message)?"
query_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query)], role=Roles.USER
)
query_with_context = QueryWithContext(query=query, history=conversation)

answer = await main_agent.answer(query_with_context)

In [ ]:
print(answer.content[0].text)

----------------